# Solve Field

This notebook can be used to determine the WCS solution of an OmegaCAM image.
It uses astrometry.net solve_field command. It is a necessary step before running scamp which needs a WCS accurate to approx 1", otherwise it fails.

Before start please be sure you can run the solve_field command from the astrometry package by tipying
/usr/local/astrometry/bin/solve-field

If you get something like
ERROR: You didn't specify any files to process.<br>
This program is part of the Astrometry.net suite.<br>
For details, visit http://astrometry.net.<br>
Git URL https://github.com/dstndstn/astrometry.net<br>
Revision 0.89, date Mon_Jan_17_07:47:26_2022_-0500.<br>

then you are okay.

This notebook will create a independent fits image for each CCD within the input mosaic. Then run solve_field on each one to find an astrometric solution. This solution will be incorporated to a new mosaic which will be use later.
Please be sure you have enough space in your hard disk, as this notebook will effectively created two new mosaic images for each input one.

Tested on my personal laptop using the index fields 4100 and 5200 series from astromatic.
Known issues:
- For some reason, the astrometry package failed during the flow of the script, but running the same command in terminal directly over the CCD fits image retrieves a solution in seconds.
- For those CCDs that have failed at retrieve a solution with astrometry, are copied directly into the final mosaic, causing some shifts between CCDs.

Lientur

In [1]:
import os
import sys

import glob
import logging
import multiprocessing

sys.path.append(os.path.join(os.path.abspath(''), '..')) # to be removed in production
from pycavi import pycavi_solve
from pycavi import pycavi_tools
from pycavi import pycavi_instrument

This will be executed when the package is imported.


In [2]:
import multiprocessing

multiprocessing.cpu_count()

8

In [3]:
# Set the reduced science dir where it will look for the reduced science images, same as previous notebooks
# For each target within the folder will check the filters before start to work on each observation.
# In this case, during the tests I consider only the images in the filter NB_659.

config_dir = os.getcwd()+'/'+'../config/'

config = pycavi_tools.read_pycavi_config(config_dir)
instrument = config['Provenance']['instrument'].lower()
reduced_science_dir = config['Directories']['reduced_science_dir']
reduced_std_dir = config['Directories']['reduced_std_dir']
log_dir = config['Directories']['project_logs_dir']

# Should read the instrument configuration file. This ensure
# proper keyword parsing.
#
instrument = pycavi_instrument.Instrument(config_dir=config_dir, instrument=instrument)

# a list with the tags of image filters. During the tests I worked only with r_SDSS and NB_659
filters_to_sol = ["BB_U_38_ESO841", "BB_V_89_ESO843", "BB_B_99_ESO842", "g_SDSS", "r_SDSS", "NB_659", "NB_OIII_8_ESO859"]

# target list will include all the observed targets within the reduced_science directory
target_list = []
for object_ in glob.glob(reduced_science_dir + "*"):
    target_list.append(object_)

for object_ in glob.glob(reduced_std_dir + "*"):
    target_list.append(object_)
    
target_list.sort()

# for each observed target will all the images which filters correspond to the ones given in filters_to_sol
for target_ in target_list:
    for filter_ in filters_to_sol:
        for observation_ in glob.glob(target_ + "/" + filter_ + "/*"):
                print(observation_)
                pycavi_solve.run_py4_solve(observation_, instrument, cpu_count=multiprocessing.cpu_count()-2, redo_astrometry=False)

# job done
logging.info("\n\n ****** Py4 Done! ******\n\n")

2024-08-12 23:00:18,273:INFO:current directory: /Users/fselman/Pictures/OCAM/MultiObject/science/CP_Pup/r_SDSS/2017-03-13T01:57:49
2024-08-12 23:00:18,275:INFO:solving image 01/10: OMEGA.2017-03-13T02:44:33.457_pre.fits
2024-08-12 23:00:18,277:INFO:wcs image already exist, skipping to the next image
2024-08-12 23:00:18,278:INFO:solving image 02/10: OMEGA.2017-03-13T02:39:07.286_pre.fits
2024-08-12 23:00:18,280:INFO:wcs image already exist, skipping to the next image
2024-08-12 23:00:18,281:INFO:solving image 03/10: OMEGA.2017-03-13T02:35:52.825_pre.fits
2024-08-12 23:00:18,283:INFO:wcs image already exist, skipping to the next image
2024-08-12 23:00:18,284:INFO:solving image 04/10: OMEGA.2017-03-13T02:40:12.356_pre.fits
2024-08-12 23:00:18,285:INFO:wcs image already exist, skipping to the next image
2024-08-12 23:00:18,287:INFO:solving image 05/10: OMEGA.2017-03-13T02:41:17.786_pre.fits
2024-08-12 23:00:18,289:INFO:wcs image already exist, skipping to the next image
2024-08-12 23:00:18

config_dir: /Users/fselman/Software/Pycavi/notebooks/../config/ instrument: omegacam
MEF.BIAS.NAME.O: OCAM_MB
MEF.BPM.NAME.O: OCAM_BPM
MEF.DOMEFF.NAME.O: OCAM_DOMEFF
MEF.SKYFF.NAME.O: OCAM_SKYFF
MEF.MFF.NAME.O: OCAM_MFF
/Users/fselman/Pictures/OCAM/MultiObject/science/CP_Pup/r_SDSS/2017-03-13T01:57:49
OMEGA.*_pre.fits
/Users/fselman/Pictures/OCAM/MultiObject/science/M67/g_SDSS/2014-12-02T06:32:53
OMEGA.*_pre.fits
/Users/fselman/Pictures/OCAM/MultiObject/science/M67/r_SDSS/2014-12-02T06:51:24
OMEGA.*_pre.fits
/Users/fselman/Pictures/OCAM/MultiObject/science/M67/NB_659/2014-12-02T07:04:51
OMEGA.*_pre.fits
/Users/fselman/Pictures/OCAM/MultiObject/std/SA101/g_SDSS/2014-12-02T05:57:25
OMEGA.*_pre.fits
/Users/fselman/Pictures/OCAM/MultiObject/std/SA101/r_SDSS/2014-12-02T05:57:25
OMEGA.*_pre.fits


2024-08-12 23:00:19,909:INFO:run_solve_field: processing extensions...
2024-08-12 23:00:19,912:INFO:run_solve_field: number of extensions: 32


Reading input file 1 of 1: "/Users/fselman/Pictures/OCAM/MultiObject/std/SA101/r_SDSS/2014-12-02T05:57:25/OMEGA.2014-12-02T06:11:38.223_pre/ESO_CCD__96_solve_field//ESO_CCD__96.fits"...
Reading input file 1 of 1: "/Users/fselman/Pictures/OCAM/MultiObject/std/SA101/r_SDSS/2014-12-02T05:57:25/OMEGA.2014-12-02T06:11:38.223_pre/ESO_CCD__68_solve_field//ESO_CCD__68.fits"...
Reading input file 1 of 1: "/Users/fselman/Pictures/OCAM/MultiObject/std/SA101/r_SDSS/2014-12-02T05:57:25/OMEGA.2014-12-02T06:11:38.223_pre/ESO_CCD__66_solve_field//ESO_CCD__66.fits"...
Running command: /usr/local/opt/astrometry-net/bin/image2pnm --infile /Users/fselman/Pictures/OCAM/MultiObject/std/SA101/r_SDSS/2014-12-02T05:57:25/OMEGA.2014-12-02T06:11:38.223_pre/ESO_CCD__96_solve_field//ESO_CCD__96.fits --uncompressed-outfile /tmp/tmp.uncompressed.7bvoFZ --outfile /tmp/tmp.ppm.2uNhE8 --ppm --mydir /usr/local/opt/astrometry-net/bin/solve-field
Running command: /usr/local/opt/astrometry-net/bin/image2pnm --infile /Users

2024-08-12 23:01:23,457:INFO:solve_field: ------------ done ----------
2024-08-12 23:01:23,485:INFO:Creating wcs mosaic file OMEGA.2014-12-02T06:11:38.223_pre.fits



2024-08-12 23:01:23,487:INFO:create_wcs_frame: file_name: OMEGA.2014-12-02T06:11:38.223_pre.fits
2024-08-12 23:01:23,489:INFO:create_wcs_frame: file exists: OMEGA.2014-12-02T06:11:38.223_pre.fits
2024-08-12 23:01:23,490:INFO:create_wcs_frame: directory exists: /Users/fselman/Pictures/OCAM/MultiObject/std/SA101/r_SDSS/2014-12-02T05:57:25/OMEGA.2014-12-02T06:11:38.223_pre
2024-08-12 23:01:23,505:INFO:create_wcs_frame: WCS file exists: /Users/fselman/Pictures/OCAM/MultiObject/std/SA101/r_SDSS/2014-12-02T05:57:25/OMEGA.2014-12-02T06:11:38.223_pre/ESO_CCD__96_solve_field//ESO_CCD__96.wcs
2024-08-12 23:01:23,519:INFO:create_wcs_frame: WCS file exists: /Users/fselman/Pictures/OCAM/MultiObject/std/SA101/r_SDSS/2014-12-02T05:57:25/OMEGA.2014-12-02T06:11:38.223_pre/ESO_CCD__65_solve_field//ESO_CCD__65.wcs
2024-08-12 23:01:23,531:IN

...

wcs_fits: /Users/fselman/Pictures/OCAM/MultiObject/std/SA101/r_SDSS/2014-12-02T05:57:25/OMEGA.2014-12-02T06:11:38.223_pre/ESO_CCD__81_solve_field//ESO_CCD__81_newwcs.fits
wcs_fits: /Users/fselman/Pictures/OCAM/MultiObject/std/SA101/r_SDSS/2014-12-02T05:57:25/OMEGA.2014-12-02T06:11:38.223_pre/ESO_CCD__82_solve_field//ESO_CCD__82_newwcs.fits
wcs_fits: /Users/fselman/Pictures/OCAM/MultiObject/std/SA101/r_SDSS/2014-12-02T05:57:25/OMEGA.2014-12-02T06:11:38.223_pre/ESO_CCD__83_solve_field//ESO_CCD__83_newwcs.fits
wcs_fits: /Users/fselman/Pictures/OCAM/MultiObject/std/SA101/r_SDSS/2014-12-02T05:57:25/OMEGA.2014-12-02T06:11:38.223_pre/ESO_CCD__84_solve_field//ESO_CCD__84_newwcs.fits
wcs_fits: /Users/fselman/Pictures/OCAM/MultiObject/std/SA101/r_SDSS/2014-12-02T05:57:25/OMEGA.2014-12-02T06:11:38.223_pre/ESO_CCD__89_solve_field//ESO_CCD__89_newwcs.fits
wcs_fits: /Users/fselman/Pictures/OCAM/MultiObject/std/SA101/r_SDSS/2014-12-02T05:57:25/OMEGA.2014-12-02T06:11:38.223_pre/ESO_CCD__90_solve_field

2024-08-12 23:01:23,856:INFO:create_wcs_frame: WCS file exists: /Users/fselman/Pictures/OCAM/MultiObject/std/SA101/r_SDSS/2014-12-02T05:57:25/OMEGA.2014-12-02T06:11:38.223_pre/ESO_CCD__86_solve_field//ESO_CCD__86.wcs
2024-08-12 23:01:23,869:INFO:create_wcs_frame: WCS file exists: /Users/fselman/Pictures/OCAM/MultiObject/std/SA101/r_SDSS/2014-12-02T05:57:25/OMEGA.2014-12-02T06:11:38.223_pre/ESO_CCD__87_solve_field//ESO_CCD__87.wcs
2024-08-12 23:01:23,887:INFO:create_wcs_frame: WCS file exists: /Users/fselman/Pictures/OCAM/MultiObject/std/SA101/r_SDSS/2014-12-02T05:57:25/OMEGA.2014-12-02T06:11:38.223_pre/ESO_CCD__88_solve_field//ESO_CCD__88.wcs
2024-08-12 23:01:23,900:INFO:create_wcs_frame: WCS file exists: /Users/fselman/Pictures/OCAM/MultiObject/std/SA101/r_SDSS/2014-12-02T05:57:25/OMEGA.2014-12-02T06:11:38.223_pre/ESO_CCD__93_solve_field//ESO_CCD__93.wcs
2024-08-12 23:01:23,914:INFO:create_wcs_frame: WCS file exists: /Users/fselman/Pictures/OCAM/MultiObject/std/SA101/r_SDSS/2014-12-02

wcs_fits: /Users/fselman/Pictures/OCAM/MultiObject/std/SA101/r_SDSS/2014-12-02T05:57:25/OMEGA.2014-12-02T06:11:38.223_pre/ESO_CCD__86_solve_field//ESO_CCD__86_newwcs.fits
wcs_fits: /Users/fselman/Pictures/OCAM/MultiObject/std/SA101/r_SDSS/2014-12-02T05:57:25/OMEGA.2014-12-02T06:11:38.223_pre/ESO_CCD__87_solve_field//ESO_CCD__87_newwcs.fits
wcs_fits: /Users/fselman/Pictures/OCAM/MultiObject/std/SA101/r_SDSS/2014-12-02T05:57:25/OMEGA.2014-12-02T06:11:38.223_pre/ESO_CCD__88_solve_field//ESO_CCD__88_newwcs.fits
wcs_fits: /Users/fselman/Pictures/OCAM/MultiObject/std/SA101/r_SDSS/2014-12-02T05:57:25/OMEGA.2014-12-02T06:11:38.223_pre/ESO_CCD__93_solve_field//ESO_CCD__93_newwcs.fits
wcs_fits: /Users/fselman/Pictures/OCAM/MultiObject/std/SA101/r_SDSS/2014-12-02T05:57:25/OMEGA.2014-12-02T06:11:38.223_pre/ESO_CCD__94_solve_field//ESO_CCD__94_newwcs.fits
wcs_fits: /Users/fselman/Pictures/OCAM/MultiObject/std/SA101/r_SDSS/2014-12-02T05:57:25/OMEGA.2014-12-02T06:11:38.223_pre/ESO_CCD__95_solve_field

2024-08-12 23:01:28,332:INFO:current directory: /Users/fselman/Pictures/OCAM/MultiObject/std/SA98/NB_659/2014-12-02T07:31:14
2024-08-12 23:01:28,334:INFO:solving image 01/01: OMEGA.2014-12-02T07:52:41.089_pre.fits
2024-08-12 23:01:28,335:INFO:wcs image already exist, skipping to the next image
2024-08-12 23:01:28,336:INFO:

 ****** Py4 Done! ******




/Users/fselman/Pictures/OCAM/MultiObject/std/SA98/NB_659/2014-12-02T07:31:14
OMEGA.*_pre.fits
